# 00a · Download the models

**Run this first.** Every other notebook loads models from your disk; this notebook fetches them once.

| What | Hugging Face ID | Approx. size | Needed by |
|---|---|---|---|
| Base model | `Qwen/Qwen2.5-0.5B-Instruct` (Apache-2.0) | 1 GB | every notebook |
| Embedding model | `BAAI/bge-small-en-v1.5` (MIT) | 130 MB | retrieval notebooks 11–13 |
| Reranker | `BAAI/bge-reranker-base` (MIT) | 1.1 GB | retrieval notebooks 11–13 |

The base model goes to `models/Qwen2.5-0.5B-Instruct/` inside this project (git-ignored). The two retrieval models go to the standard Hugging Face cache, pinned to the exact revisions in `configs/rag.yaml`, because the retrieval code loads them by ID and revision from local files only.

## Do you need a Hugging Face token?

**Not for these three models.** They are public. A token only matters if you:

- switch to a **gated** model (for example Llama or Gemma), where you must first accept its licence on the model's Hugging Face page, or
- run into anonymous download rate limits.

To use a token:

1. Create one at huggingface.co → **Settings → Access Tokens**. Choose **Read** access; nothing in this lab needs write access.
2. Give it to the lab in **one** of these ways. Never paste it into a notebook cell.
   - Copy `.env.example` to `.env` in the project root and set `HF_TOKEN=...` (`.env` is git-ignored).
   - Or run `export HF_TOKEN=...` in the terminal before starting Jupyter.
   - Or run `hf auth login` once in a terminal; the Hugging Face libraries then find the token themselves.

The next cell reports only **whether** a token was found. It never prints the token.

In [ ]:
# Put the project root on the import path so `from src import lab` works from notebooks/.
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
from pathlib import Path
from src import lab, rag

CFG = lab.cfg()
TOKEN = lab.hf_token()   # None unless HF_TOKEN is set in the environment or in .env
print("Hugging Face token found in env/.env:", TOKEN is not None)
print("Base model ID :", CFG["model"]["hf_id"])
print("Download to   :", CFG["model"]["base_path"])

## 1. The base model

`snapshot_download` fetches a model repository's files. `allow_patterns` limits it to what inference and training need: weights (`*.safetensors`), configuration, tokenizer files and the licence. If the folder already holds the weights, nothing is downloaded again.

With `token=None`, the library still uses a login saved by `hf auth login`, if you made one.

In [ ]:
from huggingface_hub import snapshot_download

base_dir = Path(CFG["model"]["base_path"])
if (base_dir / "config.json").is_file() and any(base_dir.glob("*.safetensors")):
    print("Already downloaded:", base_dir)
else:
    snapshot_download(
        repo_id=CFG["model"]["hf_id"],
        local_dir=str(base_dir),
        token=TOKEN,
        allow_patterns=["*.json", "*.safetensors", "*.txt", "LICENSE", "README.md"],
    )
    print("Downloaded to:", base_dir)

size_gb = sum(f.stat().st_size for f in base_dir.rglob("*") if f.is_file()) / 1024**3
print(f"{sum(1 for _ in base_dir.iterdir())} entries | {size_gb:.2f} GB on disk")

In [ ]:
# Confirm the download is usable without loading ~1 GB of weights: read only config and tokenizer.
from transformers import AutoConfig, AutoTokenizer
config = AutoConfig.from_pretrained(base_dir)
tok = AutoTokenizer.from_pretrained(base_dir)
print("architecture:", config.architectures, "| layers:", config.num_hidden_layers, "| vocabulary:", len(tok))
print("chat template present:", tok.chat_template is not None)

## 2. The retrieval models (only for notebooks 11–13)

Skip this cell if you are not running the retrieval notebooks. Revisions are pinned in `configs/rag.yaml`, so everyone who runs the lab embeds with exactly the same weights.

In [ ]:
DOWNLOAD_RETRIEVAL_MODELS = True   # set False to skip about 1.2 GB

RAG_CFG = rag.load_config()
if DOWNLOAD_RETRIEVAL_MODELS:
    for section in ("embedding", "rerank"):
        s = RAG_CFG[section]
        path = snapshot_download(
            repo_id=s["model_id"], revision=s["revision"], token=TOKEN,
            allow_patterns=["config.json", "*.safetensors", "model.safetensors.index.json", "tokenizer.json",
                            "tokenizer_config.json", "special_tokens_map.json", "vocab.txt", "sentencepiece.bpe.model"],
        )
        print(f"{section:9s} {s['model_id']} @ {s['revision'][:10]} -> {path}")
else:
    print("Skipped retrieval models.")

## Use a different base model

1. Change `model.hf_id` and `model.base_path` in `configs/config.yaml`, for example `Qwen/Qwen2.5-1.5B-Instruct` and `models/Qwen2.5-1.5B-Instruct`, then rerun this notebook. For a gated model, set `HF_TOKEN` first and accept the licence on its model page.
2. Or reuse a copy you already have: `export PTL_BASE_MODEL_PATH=/path/to/model` before starting Jupyter.
3. Check that `training.lora.target_modules` in the config matches the new architecture's layer names. Qwen, Llama and Mistral use `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj` and `down_proj`.
4. An adapter only works with the base model it was trained on. Retrain after switching.

Memory rule of thumb in float32: 4 bytes per parameter for the weights alone. A 1.5B model needs about 6 GB before activations; LoRA keeps the trainable part small, but the frozen base still has to fit.

**Next:** notebook 00b checks your hardware and runs the base model once.